# https://www.kaggle.com/datasets/umuttuygurr/e-commerce-fraud-detection-dataset

In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs
import altair as alt
import plotly.express as px
import plotly.graph_objects as go
import great_tables as tg
import datetime as dt
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE

In [2]:
df_path = r'/Users/zygimantas/Documents/Data_sets/transactions.csv'

In [3]:
df = pl.read_csv(df_path, infer_schema_length=10_000)

In [4]:
df.null_count()

transaction_id,user_id,account_age_days,total_transactions_user,avg_amount_user,amount,country,bin_country,channel,merchant_category,promo_used,avs_match,cvv_result,three_ds_flag,transaction_time,shipping_distance_km,is_fraud
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [5]:
df.select(
    df.select(
        cs.string().n_unique()
    )
)

country,bin_country,channel,merchant_category,transaction_time
u32,u32,u32,u32,u32
10,10,2,5,297975


In [6]:
df.collect_schema()

Schema([('transaction_id', Int64),
        ('user_id', Int64),
        ('account_age_days', Int64),
        ('total_transactions_user', Int64),
        ('avg_amount_user', Float64),
        ('amount', Float64),
        ('country', String),
        ('bin_country', String),
        ('channel', String),
        ('merchant_category', String),
        ('promo_used', Int64),
        ('avs_match', Int64),
        ('cvv_result', Int64),
        ('three_ds_flag', Int64),
        ('transaction_time', String),
        ('shipping_distance_km', Float64),
        ('is_fraud', Int64)])

In [7]:
df = df.with_columns(
    pl.col('transaction_time').str.strptime(pl.Datetime, '%Y-%m-%dT%H:%M:%SZ')
)

In [8]:
df.select(
    cs.all().shrink_dtype()
).estimated_size('mb')

13.834456443786621

In [9]:
df.select(
    cs.all()
).estimated_size('mb')

33.8412561416626

In [10]:
df = df.select(
    cs.all().shrink_dtype()
)

In [11]:
df.select(
    cs.exclude(['transaction_id'])
)

user_id,account_age_days,total_transactions_user,avg_amount_user,amount,country,bin_country,channel,merchant_category,promo_used,avs_match,cvv_result,three_ds_flag,transaction_time,shipping_distance_km,is_fraud
i16,i16,i8,f32,f32,str,str,str,str,i8,i8,i8,i8,datetime[μs],f32,i8
1,141,47,147.929993,84.75,"""FR""","""FR""","""web""","""travel""",0,1,1,1,2024-01-06 04:09:39,370.950012,0
1,141,47,147.929993,107.900002,"""FR""","""FR""","""web""","""travel""",0,0,0,0,2024-01-09 20:13:47,149.619995,0
1,141,47,147.929993,92.360001,"""FR""","""FR""","""app""","""travel""",1,1,1,1,2024-01-12 06:20:11,164.080002,0
1,141,47,147.929993,112.470001,"""FR""","""FR""","""web""","""fashion""",0,1,1,1,2024-01-15 17:00:04,397.399994,0
1,141,47,147.929993,132.910004,"""FR""","""US""","""web""","""electronics""",0,1,1,1,2024-01-17 01:27:31,935.280029,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
6000,996,45,27.93,34.07,"""ES""","""ES""","""web""","""grocery""",0,1,1,0,2024-09-29 04:40:54,218.550003,0
6000,996,45,27.93,68.559998,"""ES""","""ES""","""app""","""travel""",0,1,1,1,2024-10-03 08:49:02,185.550003,0
6000,996,45,27.93,25.02,"""ES""","""ES""","""app""","""fashion""",0,1,1,1,2024-10-26 07:40:38,33.5,0


In [12]:
df = df.with_columns(
    pl.col('transaction_time').dt.year().alias('year'),
    pl.col('transaction_time').dt.month().alias('month'),
    pl.col('transaction_time').dt.day().alias('day'),
    pl.col('transaction_time').dt.hour().alias('hour'),
    pl.col('transaction_time').dt.minute().alias('minute'),
    pl.col('transaction_time').dt.second().alias('second'),
).select(
    cs.all().shrink_dtype()
)

In [13]:
df.estimated_size('mb')

15.835136413574219

In [14]:
df.select(
    cs.exclude(['transaction_id', 'transaction_time'])
).estimated_size('mb')

12.405399322509766

In [15]:
df = df.select(
    cs.exclude(['transaction_id', 'transaction_time'])
)

In [16]:
df = df.with_columns(
    pl.col('is_fraud').cast(pl.Boolean),
)

In [17]:
df.estimated_size('mb')

12.15536880493164

In [18]:
df

user_id,account_age_days,total_transactions_user,avg_amount_user,amount,country,bin_country,channel,merchant_category,promo_used,avs_match,cvv_result,three_ds_flag,shipping_distance_km,is_fraud,year,month,day,hour,minute,second
i16,i16,i8,f32,f32,str,str,str,str,i8,i8,i8,i8,f32,bool,i16,i8,i8,i8,i8,i8
1,141,47,147.929993,84.75,"""FR""","""FR""","""web""","""travel""",0,1,1,1,370.950012,false,2024,1,6,4,9,39
1,141,47,147.929993,107.900002,"""FR""","""FR""","""web""","""travel""",0,0,0,0,149.619995,false,2024,1,9,20,13,47
1,141,47,147.929993,92.360001,"""FR""","""FR""","""app""","""travel""",1,1,1,1,164.080002,false,2024,1,12,6,20,11
1,141,47,147.929993,112.470001,"""FR""","""FR""","""web""","""fashion""",0,1,1,1,397.399994,false,2024,1,15,17,0,4
1,141,47,147.929993,132.910004,"""FR""","""US""","""web""","""electronics""",0,1,1,1,935.280029,false,2024,1,17,1,27,31
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
6000,996,45,27.93,34.07,"""ES""","""ES""","""web""","""grocery""",0,1,1,0,218.550003,false,2024,9,29,4,40,54
6000,996,45,27.93,68.559998,"""ES""","""ES""","""app""","""travel""",0,1,1,1,185.550003,false,2024,10,3,8,49,2
6000,996,45,27.93,25.02,"""ES""","""ES""","""app""","""fashion""",0,1,1,1,33.5,false,2024,10,26,7,40,38


In [19]:
df = df.with_columns(
    pl.col('promo_used').cast(pl.Boolean),
    pl.col('avs_match').cast(pl.Boolean),
    pl.col('cvv_result').cast(pl.Boolean),
    pl.col('three_ds_flag').cast(pl.Boolean),
)

In [20]:
df.estimated_size('mb')

11.15524673461914

In [21]:
df

user_id,account_age_days,total_transactions_user,avg_amount_user,amount,country,bin_country,channel,merchant_category,promo_used,avs_match,cvv_result,three_ds_flag,shipping_distance_km,is_fraud,year,month,day,hour,minute,second
i16,i16,i8,f32,f32,str,str,str,str,bool,bool,bool,bool,f32,bool,i16,i8,i8,i8,i8,i8
1,141,47,147.929993,84.75,"""FR""","""FR""","""web""","""travel""",false,true,true,true,370.950012,false,2024,1,6,4,9,39
1,141,47,147.929993,107.900002,"""FR""","""FR""","""web""","""travel""",false,false,false,false,149.619995,false,2024,1,9,20,13,47
1,141,47,147.929993,92.360001,"""FR""","""FR""","""app""","""travel""",true,true,true,true,164.080002,false,2024,1,12,6,20,11
1,141,47,147.929993,112.470001,"""FR""","""FR""","""web""","""fashion""",false,true,true,true,397.399994,false,2024,1,15,17,0,4
1,141,47,147.929993,132.910004,"""FR""","""US""","""web""","""electronics""",false,true,true,true,935.280029,false,2024,1,17,1,27,31
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
6000,996,45,27.93,34.07,"""ES""","""ES""","""web""","""grocery""",false,true,true,false,218.550003,false,2024,9,29,4,40,54
6000,996,45,27.93,68.559998,"""ES""","""ES""","""app""","""travel""",false,true,true,true,185.550003,false,2024,10,3,8,49,2
6000,996,45,27.93,25.02,"""ES""","""ES""","""app""","""fashion""",false,true,true,true,33.5,false,2024,10,26,7,40,38


In [22]:
X = df.select(
    cs.exclude('is_fraud')
)

In [23]:
y = df.get_column(
    'is_fraud'
)

In [24]:
X_numeric = df.select(
    cs.numeric()
)

In [25]:
X_categorical = df.select(
    cs.string()
)

In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [27]:
column_transformer = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(
            handle_unknown='ignore', sparse_output=False, drop='first'),
        X_categorical.columns),
        ('pass', 'passthrough', X_numeric.columns)
    ]
)

In [28]:
column_transformer

,transformers,"[('encoder', ...), ('pass', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,'first'
,sparse_output,False


In [29]:
X_encoded = column_transformer.fit_transform(X)

In [30]:
feature_names = column_transformer.named_transformers_['encoder'].get_feature_names_out()

In [32]:
numeric_columns = df.select(cs.numeric()).columns

In [33]:
all_feature_names = list(feature_names) + list(numeric_columns)

In [36]:
X_encoded = pl.DataFrame(
    X_encoded, schema=all_feature_names
)

In [37]:
X_encoded

country_ES,country_FR,country_GB,country_IT,country_NL,country_PL,country_RO,country_TR,country_US,bin_country_ES,bin_country_FR,bin_country_GB,bin_country_IT,bin_country_NL,bin_country_PL,bin_country_RO,bin_country_TR,bin_country_US,channel_web,merchant_category_fashion,merchant_category_gaming,merchant_category_grocery,merchant_category_travel,user_id,account_age_days,total_transactions_user,avg_amount_user,amount,shipping_distance_km,year,month,day,hour,minute,second
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,141.0,47.0,147.929993,84.75,370.950012,2024.0,1.0,6.0,4.0,9.0,39.0
0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,141.0,47.0,147.929993,107.900002,149.619995,2024.0,1.0,9.0,20.0,13.0,47.0
0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,141.0,47.0,147.929993,92.360001,164.080002,2024.0,1.0,12.0,6.0,20.0,11.0
0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,141.0,47.0,147.929993,112.470001,397.399994,2024.0,1.0,15.0,17.0,0.0,4.0
0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,141.0,47.0,147.929993,132.910004,935.280029,2024.0,1.0,17.0,1.0,27.0,31.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,6000.0,996.0,45.0,27.93,34.07,218.550003,2024.0,9.0,29.0,4.0,40.0,54.0
1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,6000.0,996.0,45.0,27.93,68.559998,185.550003,2024.0,10.0,3.0,8.0,49.0,2.0
1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,6000.0,996.0,45.0,27.93,25.02,33.5,2024.0,10.0,26.0,7.0,40.0,38.0


In [38]:
from sklearn.model_selection import train_test_split

In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)